In [13]:
!pip install langchain-groq tavily-python scrapegraph-py

In [16]:
import os
import json
from typing import List, Optional, TypedDict

from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from tavily import TavilyClient
from scrapegraph_py import ScrapeGraphAI

In [ ]:
GROQ_API_KEY = os.environ('GROQ_API_KEY')
TAVILY_API_KEY = os.environ('TAVILY_API_KEY')
SGAI_API_KEY = os.environ('SGAI_API_KEY')

## Setup

In [38]:
llm = ChatGroq(
        model="qwen/qwen3.8-27b",
        api_key=GROQ_API_KEY,
        temperature=0)

search_client = TavilyClient(api_key=TAVILY_API_KEY)
scrape_client = ScrapeGraphAI(api_key=SGAI_API_KEY)

output_dir = "./ai-agent-output"
os.makedirs(output_dir, exist_ok=True)

ABOUT_COMPANY = (
    "Rankyx is a company that provides AI solutions to help websites "
    "refine their search and recommendation systems."
)

## Schemas

In [39]:
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(..., min_length=1, max_length=10)


class ProductSpec(BaseModel):
    specification_name: str
    specification_value: str


class ExtractedProduct(BaseModel):
    page_url: str = ""
    product_title: str
    product_image_url: Optional[str] = None
    product_url: Optional[str] = None
    product_current_price: float
    product_original_price: Optional[float] = None
    product_discount_percentage: Optional[float] = None
    product_specs: List[ProductSpec] = []


## State

In [40]:
class State(TypedDict, total=False):
    inputs: dict
    queries: List[str]
    search_results: List[dict]
    products: List[dict]
    report_html: str


def save_json(name, data):
    with open(os.path.join(output_dir, name), "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)



## Node 1: suggest queries

In [41]:
def queries_node(state: State):
    i = state["inputs"]
    prompt = "\n".join([
        f"Rankyx is looking to buy {i['product_name']} at the best prices (value for price).",
        f"Target websites: {i['websites_list']}",
        f"The stores must sell the product in {i['country_name']}.",
        f"Generate at most {i['no_keywords']} queries, in {i['language']}.",
        "Queries must be varied and contain specific brands, types or technologies.",
        "Queries must reach a single-product ecommerce page, not a blog or listing.",
    ])
    result = llm.with_structured_output(SuggestedSearchQueries).invoke(prompt)
    queries = result.queries[: i["no_keywords"]]
    save_json("step_1_suggested_search_queries.json", {"queries": queries})
    return {"queries": queries}


## Node 2: search

In [42]:
def search_node(state: State):
    i = state["inputs"]
    seen, results = set(), []
    for q in state["queries"]:
        try:
            resp = search_client.search(q, max_results=5)
        except Exception as e:
            print(f"search failed for '{q}': {e}")
            continue
        for r in resp.get("results", []):
            if r["score"] < i["score_th"] or r["url"] in seen:
                continue
            seen.add(r["url"])
            results.append({
                "title": r["title"],
                "url": r["url"],
                "content": r["content"][:300],
                "score": r["score"],
                "search_query": q,
            })
    results.sort(key=lambda x: x["score"], reverse=True)
    save_json("step_2_search_results.json", results)
    return {"search_results": results}


## Node 3: scrape

In [43]:
def scrape_node(state: State):
    i = state["inputs"]
    products = []
    for r in state["search_results"][: i["top_recommendations_no"]]:
        try:
            resp = scrape_client.extract(
                website_url=r["url"],
                user_prompt="Extract the product details from the web page.",
                output_schema=ExtractedProduct,
            )
            data = resp.get("result") or {}
            data["page_url"] = r["url"]
            products.append(ExtractedProduct(**data).model_dump())
        except Exception as e:
            print(f"skip {r['url']}: {e}")
    save_json("step_3_search_results.json", products)
    return {"products": products}


## Node 4: report

In [44]:
def report_node(state: State):
    prompt = "\n".join([
        "Generate a professional HTML page for a procurement report. Use Bootstrap CSS (CDN).",
        f"Company context: {ABOUT_COMPANY}",
        "Sections: 1. Executive Summary 2. Introduction 3. Methodology 4. Findings (tables)",
        "5. Analysis 6. Recommendations (rank products best to worst, with notes)",
        "7. Conclusion 8. Appendices.",
        "Return only the HTML, no markdown fences.",
        "Products JSON:",
        json.dumps(state["products"], ensure_ascii=False)[:12000],
    ])
    html = llm.invoke(prompt).content.strip()
    if html.startswith("```"):
        html = html.split("\n", 1)[1].rsplit("```", 1)[0]
    with open(os.path.join(output_dir, "step_4_procurement_report.html"), "w", encoding="utf-8") as f:
        f.write(html)
    return {"report_html": html}


## Graph

In [45]:
graph = StateGraph(State)
graph.add_node("queries", queries_node)
graph.add_node("search", search_node)
graph.add_node("scrape", scrape_node)
graph.add_node("report", report_node)
graph.add_edge(START, "queries")
graph.add_edge("queries", "search")
graph.add_edge("search", "scrape")
graph.add_edge("scrape", "report")
graph.add_edge("report", END)
app = graph.compile()


In [ ]:
if __name__ == "__main__":
    app.invoke({
        "inputs": {
            "product_name": "coffee machine for the office",
            "websites_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
            "country_name": "Egypt",
            "no_keywords": 10,
            "language": "English",
            "score_th": 0.10,
            "top_recommendations_no": 10,
        }
    })
    print("Done. See ./ai-agent-output")